# Whisper Fine-Tuning for Tajik STT

## Prepared by: Franz Kingstein Nesakumar

## 1. Libraries

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from datasets import load_dataset, Audio, disable_caching
from transformers import WhisperProcessor, WhisperForConditionalGeneration, TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
import torch
import re
import evaluate
import os
import shutil

##  1. Dataset

In [3]:
from datasets import load_dataset

# Force clean cache dir to avoid broken default cache and redownload
dataset = load_dataset("google/fleurs", "tg_tj", streaming=True)

# Confirm all splits loaded
print(dataset)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/13.3k [00:00<?, ?B/s]

fleurs.py:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_shards: 1
    })
    validation: IterableDataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_shards: 1
    })
    test: IterableDataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_shards: 1
    })
})


##  2.Load Tajik dataset from FLEURS

## 3. Normalize Transcriptions

In [4]:
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s’ʼʼғқҷҳ]", "", text)  # remove special chars
    return text.strip()


## 4. Load Whisper model and processor

In [5]:
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="tajik", task="transcribe")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

## 5. Preprocess audio + text

In [17]:
def preprocess(example):
    audio = example["audio"]
    text = normalize(example["transcription"])

    # Process a single example with return_tensors="pt" to get torch tensors
    inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], text=text, return_tensors="pt", padding=True)

    # Assign input features and labels
    example["input_features"] = inputs.input_features[0]
    example["labels"] = inputs.labels[0]

    return example

# Apply preprocessing
train_data = dataset["train"].map(preprocess)

## 6. Collator for variable-length labels

In [18]:
@dataclass
class DataCollatorWhisper:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = torch.stack([torch.tensor(f["input_features"]) for f in features])
        labels = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f["labels"]) for f in features],
            batch_first=True,
            padding_value=-100,
        )
        return {"input_features": input_features, "labels": labels}

## Timer

In [19]:
from transformers import TrainerCallback, TrainerState, TrainerControl
import time

class TimeRemainingCallback(TrainerCallback):
    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        self.start_time = time.time()
        print("Training started...")

    def on_log(self, args, state: TrainerState, control: TrainerControl, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            elapsed = time.time() - self.start_time
            steps_done = state.global_step
            total_steps = args.max_steps
            if steps_done > 0:
                estimated_total = elapsed / steps_done * total_steps
                remaining = estimated_total - elapsed
                hrs, rem = divmod(remaining, 3600)
                mins, secs = divmod(rem, 60)
                print(f"Estimated time remaining: {int(hrs):02}:{int(mins):02}:{int(secs):02}")


## 7. Training setup

In [20]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/whisper-tgk-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_steps=1000,                    # Required for iterable dataset
    logging_steps=10,
    save_steps=100,
    save_strategy="steps",
    save_total_limit=3,
    learning_rate=1e-5,
    warmup_steps=100,
    fp16=torch.cuda.is_available(),
)


## 8. Train the model

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=DataCollatorWhisper(processor),
    processing_class=processor.feature_extractor,
    callbacks=[TimeRemainingCallback()],
)


    # To resume training from a checkpoint, uncomment the line below and
    # replace 'path_to_your_checkpoint' with the actual path in your Google Drive.
    # This would be the path to the specific checkpoint folder, e.g.,
    # '/content/drive/MyDrive/whisper-tgk-checkpoints/checkpoint-50'
  #trainer.train(resume_from_checkpoint='/content/drive/MyDrive/whisper-tgk-checkpoints/checkpoint-50') # Example path

    # If not resuming, start training normally
trainer.train() # Keep this commented out if you are resuming

Training started...


KeyError: 'input_features'

## 9.  Evaluate Word Error Rate (WER) (Optional)

In [ ]:
import evaluate
metric = evaluate.load("wer")

# Example test
preds = ["Салом, ман хубам"]
refs = ["Салом, ман хубам"]
print("WER:", metric.compute(predictions=preds, references=refs))

## 10. Export Your Model

In [ ]:
model.save_pretrained("./whisper-tgk")
processor.save_pretrained("./whisper-tgk")